# Hyperparameter Tuning Log

This notebook optimizes the hyperparameters for **7 machine learning models**.

**Data Source:** `Prototype.csv` (Split 80/20 for training/tuning and validation).

**Setup:**
- **Safe Mode:** `n_jobs=1` (Prevents crashes).
- **Evaluation:** Reports Accuracy, F1-Score, and Specificity for the best configuration found.

In [15]:
# 1. Imports
import pandas as pd
import numpy as np
import os
import time
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# Models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

# --- Helper: Evaluation Report ---
def evaluate_model(y_true, y_pred, model_name):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    # Calculate Specificity (Average TNR)
    cm = confusion_matrix(y_true, y_pred)
    FP = cm.sum(axis=0) - np.diag(cm) 
    FN = cm.sum(axis=1) - np.diag(cm)
    TP = np.diag(cm)
    TN = cm.sum() - (FP + FN + TP)
    with np.errstate(divide='ignore', invalid='ignore'):
        specificity = np.mean(np.nan_to_num(TN / (TN + FP)))

    print(f"\n>>> {model_name} PERFORMANCE (Test Set) <<<")
    print(f"Accuracy:    {acc*100:.2f}%")
    print(f"F1-Score:    {f1*100:.2f}%")
    print(f"Specificity: {specificity*100:.2f}%")
    print("-"*30)

print("Libraries loaded.")

Libraries loaded.


In [16]:
# 2. Data Loading (Prototype.csv + Split)
base_dir = os.path.dirname(os.getcwd())
csv_path = os.path.join(base_dir, "Prototype.csv")

if not os.path.exists(csv_path):
    print("Error: 'Prototype.csv' not found.")
else:
    print(f"Loading dataset: {csv_path}")
    df = pd.read_csv(csv_path)

    # Clean Labels (LabelEncoder ensures 0,1,2... sequence for XGBoost)
    le = LabelEncoder()
    df['prognosis'] = le.fit_transform(df['prognosis'])
    
    # Clean Data
    df.dropna(subset=['prognosis'], inplace=True)
    
    # Identify Features
    symptoms_list = [col for col in df.columns if col != 'prognosis']
    
    X = df[symptoms_list]
    y = df['prognosis'].astype(int)

    # 80/20 Split (Stratified)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    print(f"Data successfully loaded and split.")
    print(f"Training Samples: {len(X_train)}")
    print(f"Testing Samples:  {len(X_test)}")

Loading dataset: p:\Concordia_ECE\Fall_2025\Applied_ML\Model_development_tunned_models\Model_development\Prototype.csv
Data successfully loaded and split.
Training Samples: 3936
Testing Samples:  984


---

In [17]:
# --- Tuning KNN ---
print("\nStep 1: Tuning K-Nearest Neighbors...")
start = time.time()

knn_params = {
    'n_neighbors': [3, 5, 7, 9],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn_search = RandomizedSearchCV(KNeighborsClassifier(), knn_params, n_iter=10, cv=3, n_jobs=1, random_state=42)
knn_search.fit(X_train, y_train)

print(f"Done ({time.time() - start:.2f}s). Best: {knn_search.best_params_}")
evaluate_model(y_test, knn_search.best_estimator_.predict(X_test), "KNN")


Step 1: Tuning K-Nearest Neighbors...
Done (11.22s). Best: {'weights': 'distance', 'n_neighbors': 9, 'metric': 'manhattan'}

>>> KNN PERFORMANCE (Test Set) <<<
Accuracy:    98.98%
F1-Score:    98.98%
Specificity: 99.97%
------------------------------


In [18]:
# --- Tuning Decision Tree ---
print("\nStep 2: Tuning Decision Tree...")
start = time.time()

dt_params = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

dt_search = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), dt_params, n_iter=10, cv=3, n_jobs=1, random_state=42)
dt_search.fit(X_train, y_train)

print(f"Done ({time.time() - start:.2f}s). Best: {dt_search.best_params_}")
evaluate_model(y_test, dt_search.best_estimator_.predict(X_test), "Decision Tree")


Step 2: Tuning Decision Tree...
Done (1.79s). Best: {'min_samples_split': 2, 'max_depth': 20, 'criterion': 'entropy'}

>>> Decision Tree PERFORMANCE (Test Set) <<<
Accuracy:    83.33%
F1-Score:    83.41%
Specificity: 99.58%
------------------------------


In [19]:
# --- Tuning Random Forest ---
print("\nStep 3: Tuning Random Forest...")
start = time.time()

rf_params = {
    'n_estimators': [50, 100, 200],
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 10, 20, 30]
}

rf_search = RandomizedSearchCV(RandomForestClassifier(random_state=42), rf_params, n_iter=10, cv=3, n_jobs=1, random_state=42)
rf_search.fit(X_train, y_train)

print(f"Done ({time.time() - start:.2f}s). Best: {rf_search.best_params_}")
evaluate_model(y_test, rf_search.best_estimator_.predict(X_test), "Random Forest")


Step 3: Tuning Random Forest...
Done (22.94s). Best: {'n_estimators': 200, 'max_depth': 20, 'criterion': 'gini'}

>>> Random Forest PERFORMANCE (Test Set) <<<
Accuracy:    98.88%
F1-Score:    98.88%
Specificity: 99.97%
------------------------------


In [20]:
# --- Tuning Logistic Regression ---
print("\nStep 4: Tuning Logistic Regression...")
start = time.time()

lr_params = {
    'C': [0.1, 1, 10, 100],
    'penalty': ['l2'],
    'solver': ['lbfgs']
}

lr_search = GridSearchCV(LogisticRegression(random_state=42, max_iter=2000), lr_params, cv=3, n_jobs=1)
lr_search.fit(X_train, y_train)

print(f"Done ({time.time() - start:.2f}s). Best: {lr_search.best_params_}")
evaluate_model(y_test, lr_search.best_estimator_.predict(X_test), "Logistic Regression")


Step 4: Tuning Logistic Regression...
Done (2.37s). Best: {'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs'}

>>> Logistic Regression PERFORMANCE (Test Set) <<<
Accuracy:    99.49%
F1-Score:    99.49%
Specificity: 99.99%
------------------------------


In [21]:
# --- Tuning SVM ---
print("\nStep 5: Tuning SVM...")
start = time.time()

svm_params = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 0.1],
    'kernel': ['linear', 'rbf']
}

svm_search = RandomizedSearchCV(SVC(probability=True, random_state=42), svm_params, n_iter=5, cv=3, n_jobs=1, random_state=42)
svm_search.fit(X_train, y_train)

print(f"Done ({time.time() - start:.2f}s). Best: {svm_search.best_params_}")
evaluate_model(y_test, svm_search.best_estimator_.predict(X_test), "SVM")


Step 5: Tuning SVM...
Done (44.19s). Best: {'kernel': 'rbf', 'gamma': 'scale', 'C': 1}

>>> SVM PERFORMANCE (Test Set) <<<
Accuracy:    99.49%
F1-Score:    99.49%
Specificity: 99.99%
------------------------------


In [22]:
# --- Tuning Naive Bayes ---
print("\nStep 6: Tuning Naive Bayes...")
start = time.time()

nb_params = {'var_smoothing': np.logspace(0, -9, num=100)}

nb_search = GridSearchCV(GaussianNB(), nb_params, cv=3, n_jobs=1)
nb_search.fit(X_train, y_train)

print(f"Done ({time.time() - start:.2f}s). Best: {nb_search.best_params_}")
evaluate_model(y_test, nb_search.best_estimator_.predict(X_test), "Naive Bayes")


Step 6: Tuning Naive Bayes...
Done (44.67s). Best: {'var_smoothing': np.float64(1.0)}

>>> Naive Bayes PERFORMANCE (Test Set) <<<
Accuracy:    99.19%
F1-Score:    99.18%
Specificity: 99.98%
------------------------------


In [ ]:
# --- Tuning XGBoost ---
print("\nStep 7: Tuning XGBoost...")
start = time.time()

xgb_params = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 6],
    'subsample': [0.7, 0.9]
}

xgb_search = RandomizedSearchCV(
    XGBClassifier(eval_metric='mlogloss', use_label_encoder=False, random_state=42), 
    xgb_params, n_iter=10, cv=3, n_jobs=1, random_state=42
)
xgb_search.fit(X_train, y_train)

print(f"Done ({time.time() - start:.2f}s). Best: {xgb_search.best_params_}")
evaluate_model(y_test, xgb_search.best_estimator_.predict(X_test), "XGBoost")


Step 7: Tuning XGBoost...


NameError: name 'RandomizedSearchCV' is not defined